# N4 — Realistic Forecast

## Decision question

How does the contractual cash view change when receipt timing reflects the
team's selected confidence level and assigned scenario?


In [ ]:
from pathlib import Path
import json
import sys

# Find the public package locally. A fresh Colab runtime downloads the same
# participant-safe assets from the repository.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    for source_candidate in (candidate / 'src', candidate / 'CFOPackV002' / 'src'):
        if (source_candidate / 'workshop_bootstrap.py').exists():
            sys.path.insert(0, str(source_candidate))
            break

try:
    from workshop_bootstrap import bootstrap
except ImportError:
    from urllib.request import urlopen
    bootstrap_url = (
        'https://raw.githubusercontent.com/VinayaSharada/'
        'KateelLearningDemosToStudents/cfopack-v002-v2.0.0-alpha.1/CFOPackV002/src/workshop_bootstrap.py'
    )
    namespace = {}
    exec(compile(urlopen(bootstrap_url).read(), bootstrap_url, 'exec'), namespace)
    bootstrap = namespace['bootstrap']

ROOT, OUTPUT_DIR = bootstrap()
from cfopack_v002 import (
    analyze_fx,
    default_decisions,
    load_inputs,
    load_manifest,
    run_pipeline,
)
import workshop_visuals as viz
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    # Keep the notebooks runnable from a minimal local Python environment as
    # well as Colab/Jupyter. Rich notebook rendering remains the default.
    def Markdown(value):
        return value

    def display(value):
        print(value)

manifest = load_manifest(ROOT / 'config' / 'scenario_manifest.json')
decision_file = OUTPUT_DIR / 'N0_team_decisions.json'
if decision_file.exists():
    DECISIONS = json.loads(decision_file.read_text(encoding='utf-8'))
else:
    DECISIONS = default_decisions(manifest)


In [ ]:
data = load_inputs(ROOT / 'data' / 'synthetic')
viz.data_snapshot(data, OUTPUT_DIR, 'N4')


In [ ]:
summary = run_pipeline(ROOT, OUTPUT_DIR, DECISIONS)
print(f"Scenario {summary['scenario_version']} calculated for {DECISIONS['team_name']}")


## Compare contractual and realistic liquidity


In [ ]:
contractual = pd.read_csv(OUTPUT_DIR / 'N2_contractual_forecast.csv', parse_dates=['date'])
realistic = pd.read_csv(OUTPUT_DIR / 'N4_realistic_forecast.csv', parse_dates=['date'])
comparison = pd.read_csv(OUTPUT_DIR / 'N4_forecast_comparison.csv', parse_dates=['date'])
display(comparison)
viz.forecast_chart(
    realistic,
    manifest['minimum_liquidity'],
    OUTPUT_DIR,
    'N4_realistic_vs_contractual.png',
    f"{DECISIONS['forecast_view'].upper()} realistic versus contractual cash",
    comparison=contractual,
)
largest_gap = comparison.loc[comparison['scenario_gap'].idxmax()]
print(f"Largest contractual-to-realistic gap: ${largest_gap['scenario_gap']:,.0f} on Day {int(largest_gap['day'])}")


## Team decision

Is this a forecast miss, a liquidity risk, or both? Choose the cash threshold
and day that should trigger CFO escalation, and explain why.


### Before moving on

Record your interpretation in the participant workbook. Do not copy a chart
without also recording the assumption and decision it supports.
